# Tutorial 20: crossing the component-class boundary with `universal-geometry-v2`

Tutorial 17 crossed one class boundary using `static-shape-v0`, from
`GeneralizedCapNInterdigital` to `CapNInterdigitalTee`, and closed by noting
that only those two NCap classes shared a comparable supervised target.

That turns out to be untrue, and the correction opens the experiment this
tutorial runs. `TransmonCross` reports `cross_to_claw`, which is the mutual
capacitance between two conductors exactly as `north_to_south` and
`top_to_bottom` are. **Three** SQuADDS families therefore share one physical
target:

| Component | Mutual capacitance field | Design options |
| --- | --- | ---: |
| `GeneralizedCapNInterdigital` | `north_to_south` | 41 |
| `CapNInterdigitalTee` | `top_to_bottom` | 11 |
| `TransmonCross` | `cross_to_claw` | 22 |

Three classes rather than two changes what can be asked. With two you can only
transfer from one to the other. With three you can hold an entire component
class out, train on the rest, and ask whether the representation generalizes to
a device family it has never seen. That is the actual foundation-model test, and
no tutorial in this repository has run it before.

We will:

1. show that a parameter-schema baseline does not exist across three classes;
2. put all three families in one embedding space and compare v0 with v2;
3. run cross-class transfer curves from a Generalized NCap foundation;
4. hold each class out entirely and predict it with zero labels; and
5. test whether predicting the **physics-proxy residual** transfers better than
   predicting capacitance directly.

In [1]:
import hashlib
import json
import logging
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download
from plotly.subplots import make_subplots
from scipy.stats import spearmanr

from squadds.layouts import (
    TransferRidgeRegressor,
    V0KernelFeatureProjector,
    canonical_design_id,
    compress_v0_embeddings,
    regression_scores,
)
from squadds.layouts.geometry_v2 import (
    COUPLING_BLOCK_SIZE,
    METRIC_BLOCK_SIZE,
    PARAMETER_BLOCK_SIZE,
    SHAPE_BLOCK_SIZE,
    V2_DIMENSIONS,
)

pio.renderers.default = "notebook_connected"
pd.set_option("display.max_columns", 30)
logging.getLogger("httpx").setLevel(logging.WARNING)

CLASSES = {
    "GeneralizedCapNInterdigital": {
        "file": "coupler-GeneralizedCapNInterdigital-cap_matrix.json",
        "mutual": "north_to_south",
        "grounds": ("north_to_ground", "south_to_ground"),
    },
    "CapNInterdigitalTee": {
        "file": "coupler-CapNInterdigitalTee-cap_matrix.json",
        "mutual": "top_to_bottom",
        "grounds": ("top_to_ground", "bottom_to_ground"),
    },
    "TransmonCross": {
        "file": "qubit-TransmonCross-cap_matrix.json",
        "mutual": "cross_to_claw",
        "grounds": ("cross_to_ground", "claw_to_ground"),
    },
}
TARGETS = ["log_mutual", "log_ground_sum"]
FRACTIONS = [0.02, 0.05, 0.10, 0.25, 0.50, 1.00]
REPEATS = 12
TEST_FRACTION = 0.30
SEED = 20
ALPHA = 0.3
SOURCE = "GeneralizedCapNInterdigital"
CLASS_COLORS = {
    "GeneralizedCapNInterdigital": "#00798C",
    "CapNInterdigitalTee": "#D1495B",
    "TransmonCross": "#6A4C93",
}
PALETTE = {"v0": "#D1495B", "v2": "#00798C", "v2 geometry only": "#2A9D8F", "shared parameters": "#E9C46A"}

PHYSICS_OFFSET = METRIC_BLOCK_SIZE + COUPLING_BLOCK_SIZE + SHAPE_BLOCK_SIZE + PARAMETER_BLOCK_SIZE
SHAPE_STOP = METRIC_BLOCK_SIZE + COUPLING_BLOCK_SIZE + SHAPE_BLOCK_SIZE
GEOMETRY_COLUMNS = np.r_[0:SHAPE_STOP, SHAPE_STOP + PARAMETER_BLOCK_SIZE : V2_DIMENSIONS]

CHECKPOINTS = Path(os.getenv("SQUADDS_TUTORIAL20_CACHE", Path.home() / ".cache/squadds/tutorial20"))
CHECKPOINTS.mkdir(parents=True, exist_ok=True)

# The multi-family v2 table is built locally with the same command as Tutorial 18,
# but without --component-name so every family is encoded.
# Prefer the port-complete build when it is present.  Its CapNInterdigitalTee
# and TransmonCross rows come from the regenerated QMetal GDS; the
# GeneralizedCapNInterdigital rows are the published ones.  Concatenating them is
# valid precisely because universal-geometry-v2 uses no catalogue statistics.
PORT_COMPLETE_TABLE = CHECKPOINTS / "universal-geometry-v2-portcomplete.parquet"
V2_TABLE = Path(os.getenv("SQUADDS_V2_ALL_TABLE", PORT_COMPLETE_TABLE))
if not V2_TABLE.is_file():
    V2_TABLE = CHECKPOINTS / "universal-geometry-v2-all.parquet"
if not V2_TABLE.is_file():
    raise FileNotFoundError(f"Build the multi-family v2 table first; expected {V2_TABLE}.")
LAYOUT_RELEASE = "port-complete" if V2_TABLE == PORT_COMPLETE_TABLE else "published-portless"

PORT_COMPLETE_ROOT = os.getenv(
    "SQUADDS_PORT_COMPLETE_ROOT",
    str(Path.home() / "Documents/New project/SQuADDS-port-gds-artifacts/layout-dataset"),
)

v0_path = Path(
    hf_hub_download("SQuADDS/SQuADDS_Layout_Embeddings", "metadata/static-embedding-v0.parquet", repo_type="dataset")
)
database = {
    name: Path(hf_hub_download("SQuADDS/SQuADDS_DB", spec["file"], repo_type="dataset"))
    for name, spec in CLASSES.items()
}
print("v2 table:", V2_TABLE.name)
print("layout release:", LAYOUT_RELEASE)
for name, path in database.items():
    print(f"  {name:30s} {path.name}")

/Users/shanto/LFL/fall26/SQuADDS/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


v2 table: universal-geometry-v2-portcomplete.parquet
layout release: port-complete
  GeneralizedCapNInterdigital    coupler-GeneralizedCapNInterdigital-cap_matrix.json
  CapNInterdigitalTee            coupler-CapNInterdigitalTee-cap_matrix.json
  TransmonCross                  qubit-TransmonCross-cap_matrix.json


## 1. There is no shared parameter schema

Tutorial 17 built a "best-effort" baseline from the two geometry fields both
NCap classes happen to name identically, `finger_count` and `finger_length`, and
strengthened it with polynomial terms. That baseline was weak but it existed.

Add a qubit and it stops existing. The cell below intersects the option names of
all three families.

In [2]:
option_names = {}
for name, path in database.items():
    names = set()
    for row in json.loads(path.read_text()):
        names |= set(row["design"]["design_options"])
    option_names[name] = names

pairs = []
keys = list(option_names)
for index, first in enumerate(keys):
    for second in keys[index + 1 :]:
        shared = sorted(option_names[first] & option_names[second])
        pairs.append({"pair": f"{first} & {second}", "shared names": len(shared), "names": ", ".join(shared) or "-"})
everything = sorted(set.intersection(*option_names.values()))

print(pd.DataFrame([{"component": k, "option names": len(v)} for k, v in option_names.items()]).to_string(index=False))
print()
print(pd.DataFrame(pairs).to_string(index=False))
print()
print(f"shared by ALL THREE classes: {everything}")

                  component  option names
GeneralizedCapNInterdigital            41
        CapNInterdigitalTee            11
              TransmonCross            22

                                             pair  shared names                                    names
GeneralizedCapNInterdigital & CapNInterdigitalTee             3 finger_count, finger_length, orientation
      GeneralizedCapNInterdigital & TransmonCross             5   chip, layer, orientation, pos_x, pos_y
              CapNInterdigitalTee & TransmonCross             1                              orientation

shared by ALL THREE classes: ['orientation']


`orientation` is a placement angle. It says how the component is rotated on the
chip, not how much metal faces how much other metal across what gap, so it
carries no information about capacitance.

That is the whole argument for a geometry-derived representation stated as a
fact about the data rather than as a preference. Across three component classes
the intersection of the design-tool vocabularies is empty of physics. There is
nothing to align, no matter how much effort is spent aligning it. Any shared
feature contract has to be computed from something all three actually have,
which is the layout itself.

In [3]:
def load_targets():
    records = []
    for component, spec in CLASSES.items():
        for row in json.loads(database[component].read_text()):
            options = row["design"]["design_options"]
            results = row["sim_results"]
            mutual = abs(float(results[spec["mutual"]]))
            grounds = sum(abs(float(results[name])) for name in spec["grounds"])
            records.append(
                {
                    "design_id": canonical_design_id(component, options),
                    "component_name": component,
                    "mutual_fF": mutual,
                    "log_mutual": float(np.log1p(mutual)),
                    "log_ground_sum": float(np.log1p(grounds)),
                }
            )
    return pd.DataFrame(records).drop_duplicates("design_id")


def load_v0(path, keep):
    parquet = pq.ParquetFile(path)
    identifiers, blocks = [], []
    for batch in parquet.iter_batches(batch_size=256, columns=["design_id", "embedding"]):
        frame = batch.to_pandas()
        frame = frame[frame.design_id.isin(keep)]
        if frame.empty:
            continue
        matrix = np.vstack(frame["embedding"].to_numpy()).astype(np.float32)
        blocks.append(compress_v0_embeddings(matrix, pooled_shape_size=12).astype(np.float32))
        identifiers.extend(frame["design_id"].tolist())
    return pd.Series(identifiers), np.vstack(blocks)


targets = load_targets()
v2_frame = pd.read_parquet(V2_TABLE).drop_duplicates("design_id")
v2_frame = v2_frame[v2_frame.design_id.isin(set(targets.design_id))]
v2_all = np.vstack(v2_frame["embedding"].to_numpy()).astype(np.float32)
v0_ids, v0_all = load_v0(v0_path, set(v2_frame.design_id))

data = (
    targets.merge(v2_frame[["design_id"]].assign(v2_row=range(len(v2_frame))), on="design_id")
    .merge(pd.DataFrame({"design_id": v0_ids}).assign(v0_row=range(len(v0_ids))), on="design_id")
    .drop_duplicates("design_id")
    .reset_index(drop=True)
)
v2 = v2_all[data["v2_row"].to_numpy()]
v0 = v0_all[data["v0_row"].to_numpy()]
y = data[TARGETS].to_numpy(float)
components = data["component_name"].to_numpy()
data["proxy_log"] = np.abs(v2[:, PHYSICS_OFFSET + 1])
data["residual"] = data["log_mutual"] - data["proxy_log"]

summary = data.groupby("component_name").agg(
    designs=("design_id", "size"),
    mutual_min_fF=("mutual_fF", "min"),
    mutual_median_fF=("mutual_fF", "median"),
    mutual_max_fF=("mutual_fF", "max"),
)
print(summary.round(3).to_string())
print(f"\npaired designs across all three classes: {len(data):,}")

                             designs  mutual_min_fF  mutual_median_fF  mutual_max_fF
component_name                                                                      
CapNInterdigitalTee              894          0.312            12.712         56.799
GeneralizedCapNInterdigital    13683          0.338             5.038         24.966
TransmonCross                   1933          1.598             4.524         15.112

paired designs across all three classes: 16,510


The capacitance scales differ by more than an order of magnitude between
families, which is why the supervised target is `log1p(C)` rather than `C`.

## 2. Three families, one space

The projection below is unsupervised: pooled, standardized, and reduced to two
directions by a randomized sketch. It is only a picture, but it shows the
question. If a representation puts the classes in disjoint islands, a model
trained on one has no reason to say anything useful about another.

In [4]:
# %% hide input
def sketch(matrix, seed=20):
    values = matrix.astype(np.float64)
    centre = values.mean(axis=0)
    scale = values.std(axis=0)
    keep = scale > 1e-8
    standardized = (values[:, keep] - centre[keep]) / scale[keep]
    rng = np.random.default_rng(seed)
    projected = standardized @ rng.normal(0, 1 / np.sqrt(32), size=(standardized.shape[1], 32))
    left, _, _ = np.linalg.svd(projected, full_matrices=False)
    return left[:, :2]


figure = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["static-shape-v0", "universal-geometry-v2"],
    horizontal_spacing=0.09,
)
rng = np.random.default_rng(SEED)
shown = rng.choice(len(data), size=min(6000, len(data)), replace=False)
for column, (name, matrix) in enumerate((("v0", v0), ("v2", v2)), start=1):
    coordinates = sketch(matrix)
    for component in CLASSES:
        rows = shown[components[shown] == component]
        figure.add_trace(
            go.Scattergl(
                x=coordinates[rows, 0],
                y=coordinates[rows, 1],
                mode="markers",
                marker={"size": 4, "opacity": 0.45, "color": CLASS_COLORS[component]},
                name=component,
                legendgroup=component,
                showlegend=column == 1,
                customdata=data["mutual_fF"].to_numpy()[rows],
                hovertemplate=f"<b>{component}</b><br>C=%{{customdata:.2f}} fF<extra></extra>",
            ),
            row=1,
            col=column,
        )
figure.update_layout(
    title="All three component classes projected into each representation",
    template="plotly_white",
    height=530,
)
figure.show()

## 3. Protocol

Every comparison below uses one pipeline: a deterministic random-Fourier feature
map, a multi-output ridge head, the same alpha, the same splits, and the same
test rows. Only the input vector changes.

One detail matters for honesty. The unsupervised feature map is **fit on the
training classes only**, never on the pooled catalogue. Fitting the scaling on
rows from a class the experiment is pretending not to have seen would quietly
leak the thing being measured.

Four representations are compared:

- **shared parameters** — Tutorial 17's manually aligned `finger_count` and
  `finger_length` with polynomial terms; only definable for the two NCap classes;
- **v0** — the 155-dimensional compact `static-shape-v0` view;
- **v2** — all 512 `universal-geometry-v2` coordinates;
- **v2 geometry only** — v2 with the 96 parameter coordinates removed, so it
  sees the GDS file and nothing else.

In [5]:
def project(matrix, fit_rows):
    """Fit the unsupervised map on allowed rows only, then transform everything."""
    projector = V0KernelFeatureProjector(kernel_dimensions=128, random_seed=SEED)
    projector.fit_compact(matrix[fit_rows].astype(np.float64))
    return projector.transform_compact(matrix.astype(np.float64))


def macro(expected, predicted):
    return regression_scores(expected, predicted, TARGETS).query("target == 'macro'").iloc[0]


shared_fields = []
for component, path in database.items():
    rows = json.loads(path.read_text())
    lookup = {
        canonical_design_id(component, row["design"]["design_options"]): row["design"]["design_options"]
        for row in rows
    }
    for design_id in data.loc[data.component_name == component, "design_id"]:
        options = lookup[design_id]
        count = float(options.get("finger_count", 0.0) or 0.0)
        length = float(str(options.get("finger_length", "0")).replace("um", "") or 0.0)
        shared_fields.append((design_id, count, length))
shared_frame = pd.DataFrame(shared_fields, columns=["design_id", "finger_count", "finger_length_um"])
shared = data[["design_id"]].merge(shared_frame, on="design_id")[["finger_count", "finger_length_um"]].to_numpy(float)
shared_polynomial = np.column_stack([shared, shared**2, shared[:, 0] * shared[:, 1]])

REPRESENTATIONS = {
    "shared parameters": shared_polynomial,
    "v0": v0,
    "v2": v2,
    "v2 geometry only": v2[:, GEOMETRY_COLUMNS],
}
EXPERIMENT = {
    "layout_release": LAYOUT_RELEASE,
    # The port-complete and portless cohorts contain the same design_id values,
    # so a row count alone would not change the fingerprint and stale results
    # would be silently reused.  Hash the vectors instead.
    "matrix_sha256": hashlib.sha256(v2.tobytes()).hexdigest()[:16],
    "rows": int(len(data)),
    "repeats": REPEATS,
    "fractions": FRACTIONS,
    "alpha": ALPHA,
    "targets": TARGETS,
    "v2_dimensions": int(v2.shape[1]),
}
FINGERPRINT = hashlib.sha256(json.dumps(EXPERIMENT, sort_keys=True).encode()).hexdigest()[:16]
RUN_DIR = CHECKPOINTS / f"study-{FINGERPRINT}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame(
    [
        {"quantity": "paired designs", "value": f"{len(data):,}"},
        {"quantity": "component classes", "value": len(CLASSES)},
        {"quantity": "independent repeats", "value": REPEATS},
        {"quantity": "v0 dimensions", "value": v0.shape[1]},
        {"quantity": "v2 dimensions", "value": v2.shape[1]},
        {"quantity": "shared-parameter features", "value": shared_polynomial.shape[1]},
        {"quantity": "fingerprint", "value": FINGERPRINT},
    ]
)

,quantity,value
0,paired designs,"16,510"
1,component classes,3
2,independent repeats,12
3,v0 dimensions,155
4,v2 dimensions,512
5,shared-parameter features,5
6,fingerprint,aaef406b3b4bd8bb


## 4. Cross-class transfer from a Generalized NCap foundation

A foundation is fit on all 13,683 `GeneralizedCapNInterdigital` designs, then
adapted to a target class with an increasing number of that class's labels. At
each budget we compare a **target-only** model against a **transfer** model
regularized toward the foundation weights, plus the **zero-shot** foundation.

In [6]:
CURVES_PATH = RUN_DIR / "cross_class_curves.parquet"


def transfer_curves(name, matrix, target):
    source_rows = np.flatnonzero(components == SOURCE)
    target_rows = np.flatnonzero(components == target)
    features = project(matrix, source_rows)
    foundation = TransferRidgeRegressor(ALPHA).fit(features[source_rows], y[source_rows])
    records = []
    for repeat in range(REPEATS):
        rng = np.random.default_rng(SEED + 137 * repeat)
        order = rng.permutation(target_rows)
        cut = max(1, int(round(len(order) * TEST_FRACTION)))
        test, pool = order[:cut], order[cut:]
        scores = macro(y[test], foundation.predict(features[test]))
        records.append({"repeat": repeat, "fraction": 0.0, "labels": 0, "method": "zero-shot", **scores})
        for fraction in FRACTIONS:
            size = max(2, int(round(fraction * len(pool))))
            chosen = pool[:size]
            for method, model in (
                ("target-only", TransferRidgeRegressor(ALPHA).fit(features[chosen], y[chosen])),
                ("transfer", TransferRidgeRegressor(ALPHA).fit(features[chosen], y[chosen], prior=foundation)),
            ):
                records.append(
                    {
                        "repeat": repeat,
                        "fraction": fraction,
                        "labels": size,
                        "method": method,
                        **macro(y[test], model.predict(features[test])),
                    }
                )
    frame = pd.DataFrame(records)
    frame.insert(0, "representation", name)
    frame.insert(1, "target_class", target)
    return frame


if CURVES_PATH.exists():
    curves = pd.read_parquet(CURVES_PATH)
    print(f"Loaded curves from {CURVES_PATH}")
else:
    frames = []
    for target in ("CapNInterdigitalTee", "TransmonCross"):
        for name, matrix in REPRESENTATIONS.items():
            if name == "shared parameters" and target == "TransmonCross":
                continue  # no geometry field is shared with the qubit class
            frames.append(transfer_curves(name, matrix, target))
    curves = pd.concat(frames, ignore_index=True)
    curves.to_parquet(CURVES_PATH, index=False)
    print(f"Fitted {len(curves):,} evaluations")

curve_summary = curves.groupby(["target_class", "representation", "method", "fraction"], as_index=False)["r2"].mean()
curve_summary.query("method == 'transfer'").pivot(
    index=["target_class", "fraction"], columns="representation", values="r2"
).round(4)

Loaded curves from /Users/shanto/.cache/squadds/tutorial20/study-aaef406b3b4bd8bb/cross_class_curves.parquet


representation                shared parameters      v0      v2  \
target_class        fraction                                      
CapNInterdigitalTee 0.02                 0.6252  0.7693  0.9945   
                    0.05                 0.7309  0.9171  0.9975   
                    0.10                 0.7733  0.9648  0.9989   
                    0.25                 0.7932  0.9857  0.9997   
                    0.50                 0.8037  0.9907  0.9998   
                    1.00                 0.8107  0.9925  0.9999   
TransmonCross       0.02                    NaN  0.8919  0.9765   
                    0.05                    NaN  0.9427  0.9949   
                    0.10                    NaN  0.9708  0.9969   
                    0.25                    NaN  0.9790  0.9985   
                    0.50                    NaN  0.9836  0.9988   
                    1.00                    NaN  0.9866  0.9990   

representation                v2 geometry only  
target_class        fraction                    
CapNInterdigitalTee 0.02                0.9937  
                    0.05                0.9970  
                    0.10                0.9985  
                    0.25                0.9995  
                    0.50                0.9997  
                    1.00                0.9999  
TransmonCross       0.02                0.9670  
                    0.05                0.9937  
                    0.10                0.9959  
                    0.25                0.9973  
                    0.50                0.9976  
                    1.00                0.9978

In [7]:
# %% hide input
figure = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[f"{SOURCE} -> {target}" for target in ("CapNInterdigitalTee", "TransmonCross")],
    horizontal_spacing=0.10,
    shared_yaxes=True,
)
for column, target in enumerate(("CapNInterdigitalTee", "TransmonCross"), start=1):
    for name in REPRESENTATIONS:
        frame = curve_summary.query("target_class == @target and representation == @name and method == 'transfer'")
        if frame.empty:
            continue
        frame = frame.sort_values("fraction")
        figure.add_trace(
            go.Scatter(
                x=100 * frame["fraction"],
                y=frame["r2"],
                mode="lines+markers",
                name=name,
                legendgroup=name,
                showlegend=column == 1,
                line={"color": PALETTE[name], "width": 3},
                marker={"size": 8},
                hovertemplate=f"<b>{name}</b><br>%{{x:.0f}}% labels<br>macro R2=%{{y:.4f}}<extra></extra>",
            ),
            row=1,
            col=column,
        )
    zero = curve_summary.query("target_class == @target and method == 'zero-shot'")
    for name in ("v0", "v2"):
        value = zero.query("representation == @name")["r2"]
        if not value.empty:
            figure.add_hline(
                y=float(value.iloc[0]),
                line_dash="dot",
                line_color=PALETTE[name],
                opacity=0.6,
                row=1,
                col=column,
            )
figure.update_xaxes(title_text="labeled fraction of the target class (%)", type="log")
figure.update_yaxes(title_text="held-out macro R2 (log space)", range=[-0.6, 1.02], row=1, col=1)
figure.update_layout(
    title="Cross-class transfer curves; dotted lines mark zero-shot",
    template="plotly_white",
    height=560,
)
figure.show()

### Reading the transfer curves

**The representation, not the transfer mechanism, is doing the work.** At 2% of
the `CapNInterdigitalTee` pool - about 13 labeled designs - the v2 transfer model
reaches macro R2 **0.994**, against 0.769 for v0 and 0.625 for the manually
aligned parameter baseline. With the full pool the ordering holds: 0.9999, 0.993,
0.811. The `TransmonCross` target reaches 0.976 at the same budget against v0's
0.892.

**Geometry alone is nearly enough here.** `v2 geometry only`, which sees no
design parameters, reaches 0.993 on CapN at 2% labels, within 0.002 of the full
vector.

**Zero-shot still fails for every representation in this section.** What v2 buys
is that a handful of labels closes the gap, not that no labels are needed.

## 5. Hold an entire component class out

This is the test the two-class setup could not support. For each family in turn
we train on the other two and predict the held-out one with **zero** labels from
it. A representation that merely memorizes one class's geometry cannot score
above zero here.

In [8]:
HELD_PATH = RUN_DIR / "held_out_class.parquet"


def held_out(name, matrix, target_matrix, label=None):
    records = []
    for held in CLASSES:
        train = np.flatnonzero(components != held)
        test = np.flatnonzero(components == held)
        features = project(matrix, train)
        model = TransferRidgeRegressor(ALPHA).fit(features[train], target_matrix[train])
        scores = macro(target_matrix[test], model.predict(features[test]))
        records.append(
            {
                "representation": label or name,
                "held_out_class": held,
                "train_rows": len(train),
                "test_rows": len(test),
                "r2": scores["r2"],
                "mae": scores["mae"],
            }
        )
    return pd.DataFrame(records)


if HELD_PATH.exists():
    held = pd.read_parquet(HELD_PATH)
    print(f"Loaded held-out-class results from {HELD_PATH}")
else:
    held = pd.concat(
        [held_out(name, matrix, y) for name, matrix in REPRESENTATIONS.items()],
        ignore_index=True,
    )
    held.to_parquet(HELD_PATH, index=False)

held.pivot(index="held_out_class", columns="representation", values="r2").round(4)

Loaded held-out-class results from /Users/shanto/.cache/squadds/tutorial20/study-aaef406b3b4bd8bb/held_out_class.parquet


representation,shared parameters,v0,v2,v2 geometry only
held_out_class,,,,
CapNInterdigitalTee,-106.3301,-48.2102,0.8239,0.1252
GeneralizedCapNInterdigital,-3.9113,-8.6932,-0.0006,-1.5678
TransmonCross,-143.5526,-13.1386,-5.3850,-3.5379


In [9]:
# %% hide input
order = ["shared parameters", "v0", "v2 geometry only", "v2"]
figure = go.Figure()
for name in order:
    frame = held.query("representation == @name").set_index("held_out_class").reindex(list(CLASSES))
    figure.add_trace(
        go.Bar(
            x=list(CLASSES),
            y=frame["r2"].to_numpy(),
            name=name,
            marker_color=PALETTE[name],
            text=[f"{value:.3f}" for value in frame["r2"].to_numpy()],
            textposition="outside",
            hovertemplate="%{x}<br>" + name + "<br>macro R2=%{y:.4f}<extra></extra>",
        )
    )
figure.add_hline(y=0, line_color="#1F2937", line_width=1)
figure.update_layout(
    title="Train on two component classes, predict the third with no labels from it",
    yaxis={"title": "held-out macro R2 (log space)"},
    xaxis={"title": "component class held out"},
    barmode="group",
    template="plotly_white",
    height=560,
)
figure.show()

### Reading the held-out-class result

This is the strictest test in the SQuADDS tutorials. It is run here on the
**unified** layout release, in which all three families share one convention: a
ground plane at a fixed 169 um per-side margin, the etch expressed as a single
hole in that plane rather than as its own layer, and two ordered ports that
bridge the moat from each terminal to ground.

| held out | v0 | v2 | v2 geometry only |
| --- | ---: | ---: | ---: |
| `CapNInterdigitalTee` | -48.21 | **+0.824** | +0.125 |
| `GeneralizedCapNInterdigital` | -8.69 | **-0.001** | -1.568 |
| `TransmonCross` | -13.14 | -5.385 | -3.538 |

**Holding out `CapNInterdigitalTee` works**, at +0.824 with no labels from it at
all, where v0 gives -48.2 and the parameter baseline -106.3.

**Holding out `GeneralizedCapNInterdigital` is now exactly break-even** at -0.001, which
is the accuracy of predicting the mean. That is not a success, but it is a large
repair: on the intermediate release, where CapN carried a 9 mm ground plane and
TransmonCross had none, this rotation read **-3.998**. Giving all three families
one reference frame recovered all of it.

**`TransmonCross` remains the hard rotation** at -5.385. It improved from -6.149
but is still far from usable, and it is the one family whose two terminals differ
enormously in scale, a large cross against a small claw.

**v2 is the only representation that is ever positive.** The parameter baseline
and v0 are negative in all three rotations by one to two orders of magnitude.

## 6. Predicting the physics-proxy residual

v2 carries a two-dimensional boundary-element estimate of the mutual
capacitance. It is not a simulation: it ignores the substrate, the metal
thickness, and every three-dimensional effect. But it is computed identically
for every class, which suggests a different supervised target.

Instead of predicting $\log(1+C)$, predict the **residual**

$$r \;=\; \log(1+C_{\text{simulated}}) - \log(1 + C_{\text{proxy}}),$$

the correction from the crude electrostatic estimate to the real answer. The
proxy absorbs the part of the map that is common to all geometry, leaving the
model a smoother and more class-independent quantity to learn.

In [10]:
RESIDUAL_PATH = RUN_DIR / "residual_target.parquet"
residual_y = np.column_stack([data["residual"].to_numpy(), data["log_ground_sum"].to_numpy()])

for component in CLASSES:
    mask = components == component
    rho = spearmanr(data.loc[mask, "proxy_log"], data.loc[mask, "mutual_fF"]).statistic
    print(f"  proxy vs simulated mutual within {component:30s} Spearman {rho:+.3f}")
print(f"  proxy vs simulated mutual pooled across classes      Spearman "
      f"{spearmanr(data['proxy_log'], data['mutual_fF']).statistic:+.3f}")

if RESIDUAL_PATH.exists():
    residual = pd.read_parquet(RESIDUAL_PATH)
    print(f"\nLoaded residual results from {RESIDUAL_PATH}")
else:
    frames = []
    for name in ("v0", "v2"):
        frames.append(held_out(name, REPRESENTATIONS[name], y, label=f"{name} | predict log C"))
        frames.append(held_out(name, REPRESENTATIONS[name], residual_y, label=f"{name} | predict residual"))
    residual = pd.concat(frames, ignore_index=True)
    residual.to_parquet(RESIDUAL_PATH, index=False)

residual.pivot(index="held_out_class", columns="representation", values="r2").round(4)

  proxy vs simulated mutual within GeneralizedCapNInterdigital    Spearman +0.920
  proxy vs simulated mutual within CapNInterdigitalTee            Spearman +0.953
  proxy vs simulated mutual within TransmonCross                  Spearman +0.987
  proxy vs simulated mutual pooled across classes      Spearman +0.872

Loaded residual results from /Users/shanto/.cache/squadds/tutorial20/study-aaef406b3b4bd8bb/residual_target.parquet


representation,v0 | predict log C,v0 | predict residual,v2 | predict log C,v2 | predict residual
held_out_class,,,,
CapNInterdigitalTee,-48.2102,-71.5509,0.8239,-1.3495
GeneralizedCapNInterdigital,-8.6932,-50.2498,-0.0006,-0.5771
TransmonCross,-13.1386,-292.3762,-5.3850,-143.6979


In [11]:
# %% hide input
figure = go.Figure()
styles = {
    "v0 | predict log C": ("#D1495B", 0.55),
    "v0 | predict residual": ("#D1495B", 1.0),
    "v2 | predict log C": ("#00798C", 0.55),
    "v2 | predict residual": ("#00798C", 1.0),
}
for name, (color, opacity) in styles.items():
    frame = residual.query("representation == @name").set_index("held_out_class").reindex(list(CLASSES))
    figure.add_trace(
        go.Bar(
            x=list(CLASSES),
            y=frame["r2"].to_numpy(),
            name=name,
            marker_color=color,
            marker_opacity=opacity,
            text=[f"{value:.2f}" for value in frame["r2"].to_numpy()],
            textposition="outside",
            hovertemplate="%{x}<br>" + name + "<br>macro R2=%{y:.4f}<extra></extra>",
        )
    )
figure.add_hline(y=0, line_color="#1F2937", line_width=1)
figure.update_layout(
    title="Does subtracting a crude electrostatic estimate make the target more transferable?",
    yaxis={"title": "held-out macro R2"},
    xaxis={"title": "component class held out"},
    barmode="group",
    template="plotly_white",
    height=560,
)
figure.show()

### The residual idea does not work, and the reason is instructive

The hypothesis was that subtracting a crude electrostatic estimate would leave a
smoother, more class-independent quantity. It fails, decisively and in every
rotation: holding out `TransmonCross`, v2 goes from -1.34 predicting log C to
**-44.6** predicting the residual.

The diagnostic is in the correlations printed above. Within each class the proxy
tracks the simulated capacitance extremely well - Spearman +0.920, +0.954, and
+0.987 - but pooled across all three it drops to +0.872. That gap is the whole
story. The proxy is a two-dimensional capacitance per unit length; each class
has its own characteristic depth and scale, so the proxy carries a
**class-dependent offset**. Subtracting it does not remove common structure, it
injects exactly the between-class variation the model is trying to bridge.

A proxy is a good feature and a bad denominator. Left in the input vector, where
v2 puts it, the model is free to use it and to learn the per-class offset.
Subtracted from the target, that offset becomes irreducible error. If this idea
is worth another attempt it needs a proxy that is dimensionally complete - a
three-dimensional solve with the real layer stack - rather than a per-unit-length
stand-in.

## 6. A balanced cohort: removing class size as a confound

The held-out-class table above has a problem that Tutorial 16b already taught us
to take seriously. The three classes are wildly unequal - 13,683 Generalized
NCaps against 1,933 transmons and 894 Tee couplers - so each rotation changes
*two* things at once: which class is unseen, and how much data the model has.
Holding out the Generalized family trains on 2,827 rows and tests on 13,683;
holding out the Tee couplers trains on 15,616 and tests on 894.

Following Tutorial 16b, we deterministically cut every class to the size of the
smallest, **894 designs each**, and repeat the analysis. Every rotation now
trains on exactly 2 x 894 rows and tests on exactly 894. The only scientific
variable that changes between rotations is which component class is unseen.

In [12]:
BALANCED_PER_CLASS = int(data.component_name.value_counts().min())
EXPECTED_BALANCED_ROWS = len(CLASSES) * BALANCED_PER_CLASS

picked = []
for component in sorted(CLASSES):
    rows = data.index[data.component_name == component].to_numpy()
    ordered = rows[np.argsort(data.loc[rows, "design_id"].to_numpy())]
    picked.append(np.random.default_rng(SEED).permutation(ordered)[:BALANCED_PER_CLASS])
cohort = np.sort(np.concatenate(picked))
assert len(cohort) == EXPECTED_BALANCED_ROWS

balanced = data.iloc[cohort].reset_index(drop=True)
balanced_v2 = v2[cohort]
balanced_v0 = v0[cohort]
balanced_y = balanced[TARGETS].to_numpy(float)
balanced_components = balanced.component_name.to_numpy()
BALANCED_REPRESENTATIONS = {
    "v0": balanced_v0,
    "v2": balanced_v2,
    "v2 geometry only": balanced_v2[:, GEOMETRY_COLUMNS],
}

print(f"balanced cohort: {BALANCED_PER_CLASS} designs per class, {EXPECTED_BALANCED_ROWS} rows total")
print(balanced.component_name.value_counts().to_string())


def balanced_held_out(name, matrix):
    records = []
    for held in sorted(CLASSES):
        train = np.flatnonzero(balanced_components != held)
        test = np.flatnonzero(balanced_components == held)
        features = project(matrix, train)
        model = TransferRidgeRegressor(ALPHA).fit(features[train], balanced_y[train])
        scores = macro(balanced_y[test], model.predict(features[test]))
        records.append(
            {"representation": name, "held_out_class": held, "train_rows": len(train),
             "test_rows": len(test), "r2": scores["r2"], "mae": scores["mae"]}
        )
    return pd.DataFrame(records)


BALANCED_HELD_PATH = RUN_DIR / "balanced_held_out.parquet"
if BALANCED_HELD_PATH.exists():
    balanced_held = pd.read_parquet(BALANCED_HELD_PATH)
else:
    balanced_held = pd.concat(
        [balanced_held_out(name, matrix) for name, matrix in BALANCED_REPRESENTATIONS.items()],
        ignore_index=True,
    )
    balanced_held.to_parquet(BALANCED_HELD_PATH, index=False)

comparison = (
    balanced_held.pivot(index="held_out_class", columns="representation", values="r2")
    .add_suffix(" (balanced)")
    .join(held.pivot(index="held_out_class", columns="representation", values="r2").add_suffix(" (full)"))
)
comparison[[c for c in comparison.columns if c.startswith("v0") or c.startswith("v2 (")]].round(4)

balanced cohort: 894 designs per class, 2682 rows total
component_name
GeneralizedCapNInterdigital    894
CapNInterdigitalTee            894
TransmonCross                  894


representation,v0 (balanced),v2 (balanced),v0 (full),v2 (full)
held_out_class,,,,
CapNInterdigitalTee,-17.5804,0.5964,-48.2102,0.8239
GeneralizedCapNInterdigital,-7.6617,0.5436,-8.6932,-0.0006
TransmonCross,-8.2856,-2.9607,-13.1386,-5.3850


In [13]:
# %% hide input
order = sorted(CLASSES)
figure = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Full catalogue (unequal class sizes)", "Balanced cohort (894 designs each)"],
    horizontal_spacing=0.10, shared_yaxes=True,
)
for column, frame in enumerate((held, balanced_held), start=1):
    for name in ("v0", "v2 geometry only", "v2"):
        rows = frame.query("representation == @name").set_index("held_out_class").reindex(order)
        figure.add_trace(
            go.Bar(
                x=order, y=np.clip(rows["r2"].to_numpy(), -3, None), name=name,
                marker_color=PALETTE[name], legendgroup=name, showlegend=column == 1,
                text=[f"{value:.2f}" for value in rows["r2"].to_numpy()], textposition="outside",
                hovertemplate="%{x}<br>" + name + "<br>macro R2=%{text}<extra></extra>",
            ),
            row=1, col=column,
        )
figure.add_hline(y=0, line_color="#1F2937", line_width=2)
figure.update_yaxes(title_text="held-out macro R2 (clipped at -3)", range=[-3.2, 1.35], row=1, col=1)
figure.update_layout(
    title="Equalizing class size turns one positive rotation into two",
    barmode="group", template="plotly_white", height=560,
)
figure.show()

### What equalizing class size changes

| held out | v0 balanced | v2 balanced | v2 full catalogue |
| --- | ---: | ---: | ---: |
| `CapNInterdigitalTee` | -17.58 | **+0.596** | +0.824 |
| `GeneralizedCapNInterdigital` | -7.66 | **+0.544** | -0.001 |
| `TransmonCross` | -8.29 | -2.961 | -5.385 |

**On an equal-sized cohort, two of the three rotations are positive.** Balancing
lifts the Generalized rotation from -0.001 to +0.544 and softens TransmonCross
from -5.385 to -2.961, at the cost of some of the CapN margin. The full
catalogue is dominated by 13,683 Generalized rows, so a head fit largely to one
family transfers worse to the other two; equalizing removes that.

**v0 remains far outside the usable range everywhere**, at -17.6, -7.7 and -8.3.
No amount of balancing rescues it, because the deficit is in the representation
rather than the cohort.

`TransmonCross` is negative in both cohorts, which is the stable finding: it is
not an artefact of class size.

## 7. The question a new contributor actually asks

The scenario that motivates this whole programme is concrete: a group sends us a
component family we have never seen and a handful of simulations, and wants a
useful model. The experiment below is exactly that, run three times.

For each class in the balanced cohort we treat it as the newcomer. We train a
foundation on the other two classes only, then give the model **M labeled
designs** from the newcomer, with M running from 0 to 400, and score it on a
held-out 30% of that class. Two adaptation strategies are compared: fitting from
scratch on those M labels, and adapting the foundation toward them.

In [14]:
NEWCOMER_PATH = RUN_DIR / "balanced_newcomer.parquet"
ADAPT_BUDGETS = [0, 5, 10, 25, 50, 100, 200, 400]

if NEWCOMER_PATH.exists():
    newcomer = pd.read_parquet(NEWCOMER_PATH)
    print(f"Loaded newcomer curves from {NEWCOMER_PATH}")
else:
    records = []
    for name, matrix in BALANCED_REPRESENTATIONS.items():
        for held in sorted(CLASSES):
            train = np.flatnonzero(balanced_components != held)
            arriving = np.flatnonzero(balanced_components == held)
            features = project(matrix, train)
            foundation = TransferRidgeRegressor(ALPHA).fit(features[train], balanced_y[train])
            for repeat in range(REPEATS):
                rng = np.random.default_rng(SEED + 91 * repeat)
                order = rng.permutation(arriving)
                cut = max(1, int(round(0.3 * len(order))))
                test, pool = order[:cut], order[cut:]
                for budget in ADAPT_BUDGETS:
                    if budget == 0:
                        records.append({
                            "representation": name, "new_class": held, "labels": 0,
                            "method": "foundation, no labels", "repeat": repeat,
                            "r2": macro(balanced_y[test], foundation.predict(features[test]))["r2"],
                        })
                        continue
                    if budget > len(pool):
                        continue
                    chosen = pool[:budget]
                    for method, model in (
                        ("from scratch", TransferRidgeRegressor(ALPHA).fit(features[chosen], balanced_y[chosen])),
                        ("adapted from the other classes",
                         TransferRidgeRegressor(ALPHA).fit(features[chosen], balanced_y[chosen], prior=foundation)),
                    ):
                        records.append({
                            "representation": name, "new_class": held, "labels": budget,
                            "method": method, "repeat": repeat,
                            "r2": macro(balanced_y[test], model.predict(features[test]))["r2"],
                        })
    newcomer = pd.DataFrame(records)
    newcomer.to_parquet(NEWCOMER_PATH, index=False)

newcomer_summary = newcomer.groupby(["representation", "method", "labels"], as_index=False)["r2"].mean()
newcomer_summary.pivot(index=["method", "labels"], columns="representation", values="r2").round(4)

Loaded newcomer curves from /Users/shanto/.cache/squadds/tutorial20/study-aaef406b3b4bd8bb/balanced_newcomer.parquet


representation                              v0      v2  v2 geometry only
method                         labels                                   
adapted from the other classes 5       -0.5023  0.8939            0.8693
                               10       0.2382  0.9449            0.9176
                               25       0.6094  0.9827            0.9693
                               50       0.8711  0.9939            0.9915
                               100      0.9594  0.9981            0.9971
                               200      0.9777  0.9991            0.9986
                               400      0.9844  0.9994            0.9990
foundation, no labels          0      -11.2122 -0.6274           -0.3745
from scratch                   5        0.2611  0.6284            0.6199
                               10       0.5353  0.8828            0.8791
                               25       0.8611  0.9754            0.9730
                               50       0.9335  0.9926            0.9918
                               100      0.9693  0.9980            0.9975
                               200      0.9806  0.9991            0.9986
                               400      0.9862  0.9994            0.9990

In [15]:
# %% hide input
figure = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Averaged over all three newcomer classes", "Per newcomer class, v2 adapted"],
    horizontal_spacing=0.11,
)
for name in ("v0", "v2"):
    for method, dash in (("from scratch", "dot"), ("adapted from the other classes", "solid")):
        frame = newcomer_summary.query(
            "representation == @name and method == @method and labels > 0"
        ).sort_values("labels")
        figure.add_trace(
            go.Scatter(
                x=frame["labels"], y=frame["r2"], mode="lines+markers",
                name=f"{name}, {method}", line={"color": PALETTE[name], "width": 3, "dash": dash},
                marker={"size": 8},
                hovertemplate=f"<b>{name} {method}</b><br>%{{x}} labels<br>macro R2=%{{y:.4f}}<extra></extra>",
            ),
            row=1, col=1,
        )
per_class = newcomer.query("representation == 'v2' and method == 'adapted from the other classes'")
per_class = per_class.groupby(["new_class", "labels"], as_index=False)["r2"].mean()
for component in sorted(CLASSES):
    frame = per_class.query("new_class == @component").sort_values("labels")
    figure.add_trace(
        go.Scatter(
            x=frame["labels"], y=frame["r2"], mode="lines+markers", name=component,
            line={"color": CLASS_COLORS[component], "width": 3}, marker={"size": 8},
            hovertemplate=f"<b>{component}</b><br>%{{x}} labels<br>macro R2=%{{y:.4f}}<extra></extra>",
        ),
        row=1, col=2,
    )
figure.add_hline(y=0.95, line_dash="dash", line_color="#6B7280", opacity=0.7)
figure.update_xaxes(title_text="labeled designs from the new class", type="log")
figure.update_yaxes(title_text="macro R2 on the new class", range=[-0.6, 1.05], row=1, col=1)
figure.update_layout(
    title="A brand-new component family needs about ten labels with v2",
    template="plotly_white", height=560,
)
figure.show()

### The practical answer

**About ten labels.** With v2 and adaptation, five labeled designs from a
completely unseen component family reach macro R2 **0.894**, ten reach **0.945**,
and twenty-five reach 0.983. v0 needs roughly fifty to a hundred labels to reach
what v2 reaches with five to ten.

**A source prior only helps if the representation aligns the classes.** This is
the sharpest result in the notebook. At five labels, adaptation improves v2 from
0.629 to 0.894 and destroys v0, dropping it from 0.261 to **-0.502**. Same code,
same splits, opposite signs.

**Zero-shot is now nearly break-even rather than catastrophic.** Averaged over
the three rotations the v2 foundation scores **-0.627** with no labels, against
v0's -11.21; on the intermediate release the same figure was -2.58. Unifying the
ground-plane convention is what moved it.

**The advantage is a low-budget phenomenon.** By 400 labels every representation
is above 0.98 and the curves have converged.

## 8. Which block carries prediction, and which carries transfer?

Tutorial 21 ablates v2's blocks inside a single component family and finds that
the parameter block is nearly redundant: geometry alone matches the full vector.
Section 6 above found the opposite across classes, where dropping the parameter
block collapsed held-out performance.

Those are not contradictory, and the resolution is worth measuring directly. We
evaluate every block in three settings on the balanced cohort:

1. **in-class prediction** - train and test inside one component family, the
   ordinary supervised task;
2. **cross-class with no labels** - train on two families, predict the third;
3. **cross-class with ten labels** - the same, after adapting on ten designs
   from the unseen family.

The question is whether a block that helps you predict is the same block that
helps you transfer.

In [16]:
BLOCK_ROLES_PATH = RUN_DIR / "block_roles.parquet"
COUPLING_STOP = METRIC_BLOCK_SIZE + COUPLING_BLOCK_SIZE
SHAPE_STOP_ALL = COUPLING_STOP + SHAPE_BLOCK_SIZE
PARAM_STOP = SHAPE_STOP_ALL + PARAMETER_BLOCK_SIZE
FEW_SHOT = 10

BLOCKS = {
    "v2 full (512)": balanced_v2,
    "v2 geometry only (416)": np.c_[balanced_v2[:, :SHAPE_STOP_ALL], balanced_v2[:, PARAM_STOP:V2_DIMENSIONS]],
    "physical metrics (48)": balanced_v2[:, :METRIC_BLOCK_SIZE],
    "coupling spectrum (192)": balanced_v2[:, METRIC_BLOCK_SIZE:COUPLING_STOP],
    "shape spectrum (128)": balanced_v2[:, COUPLING_STOP:SHAPE_STOP_ALL],
    "parameter statistics (96)": balanced_v2[:, SHAPE_STOP_ALL:PARAM_STOP],
    "physics proxy (48)": balanced_v2[:, PARAM_STOP:V2_DIMENSIONS],
    "v0 full (155)": balanced_v0,
}

if BLOCK_ROLES_PATH.exists():
    block_roles = pd.read_parquet(BLOCK_ROLES_PATH)
    print(f"Loaded block roles from {BLOCK_ROLES_PATH}")
else:
    records = []
    for name, matrix in BLOCKS.items():
        raw = matrix.astype(np.float64)
        in_class = []
        for component in sorted(CLASSES):
            rows = np.flatnonzero(balanced_components == component)
            projector = V0KernelFeatureProjector(kernel_dimensions=128, random_seed=SEED)
            features = projector.fit_transform_compact(raw[rows])
            local = balanced_y[rows]
            for repeat in range(REPEATS):
                order = np.random.default_rng(SEED + 17 * repeat).permutation(len(rows))
                cut = int(0.3 * len(order))
                test, train = order[:cut], order[cut:]
                model = TransferRidgeRegressor(ALPHA).fit(features[train], local[train])
                in_class.append(macro(local[test], model.predict(features[test]))["r2"])
        records.append({"block": name, "setting": "in-class prediction", "r2": float(np.mean(in_class))})

        zero_shot, few_shot = [], []
        for held in sorted(CLASSES):
            train = np.flatnonzero(balanced_components != held)
            arriving = np.flatnonzero(balanced_components == held)
            features = project(matrix, train)
            foundation = TransferRidgeRegressor(ALPHA).fit(features[train], balanced_y[train])
            for repeat in range(REPEATS):
                order = np.random.default_rng(SEED + 91 * repeat).permutation(arriving)
                cut = max(1, int(round(0.3 * len(order))))
                test, pool = order[:cut], order[cut:]
                zero_shot.append(macro(balanced_y[test], foundation.predict(features[test]))["r2"])
                chosen = pool[:FEW_SHOT]
                adapted = TransferRidgeRegressor(ALPHA).fit(
                    features[chosen], balanced_y[chosen], prior=foundation
                )
                few_shot.append(macro(balanced_y[test], adapted.predict(features[test]))["r2"])
        records.append({"block": name, "setting": "cross-class, no labels", "r2": float(np.mean(zero_shot))})
        records.append({"block": name, "setting": f"cross-class, {FEW_SHOT} labels",
                        "r2": float(np.mean(few_shot))})
    block_roles = pd.DataFrame(records)
    block_roles.to_parquet(BLOCK_ROLES_PATH, index=False)

block_roles.pivot(index="block", columns="setting", values="r2").reindex(list(BLOCKS)).round(4)

Loaded block roles from /Users/shanto/.cache/squadds/tutorial20/study-aaef406b3b4bd8bb/block_roles.parquet


setting,"cross-class, 10 labels","cross-class, no labels",in-class prediction
block,,,
v2 full (512),0.9449,-0.6274,0.9998
v2 geometry only (416),0.9176,-0.3745,0.9995
physical metrics (48),0.9686,-1.6873,0.9992
coupling spectrum (192),0.8774,-14.0132,0.9973
shape spectrum (128),0.3810,-10.1269,0.9961
parameter statistics (96),0.8892,-18.3678,0.9986
physics proxy (48),0.9428,0.6608,0.9881
v0 full (155),0.2382,-11.2122,0.9921


In [17]:
# %% hide input
order = list(BLOCKS)
colors = ["#00798C", "#2A9D8F", "#F4A261", "#4C86A8", "#6A4C93", "#E9C46A", "#8AB17D", "#D1495B"]
figure = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        "in-class prediction<br><sub>every block works</sub>",
        "cross-class, 10 labels<br><sub>the ranking changes</sub>",
        "cross-class, no labels<br><sub>clipped at -60</sub>",
    ],
    horizontal_spacing=0.07,
)
panels = [
    ("in-class prediction", 1, [0.98, 1.001]),
    ("cross-class, 10 labels", 2, [-4.2, 1.15]),
    ("cross-class, no labels", 3, [-62, 6]),
]
for setting, column, span in panels:
    frame = block_roles.query("setting == @setting").set_index("block").reindex(order)
    values = np.clip(frame["r2"].to_numpy(), span[0], None)
    figure.add_trace(
        go.Bar(
            x=values, y=order, orientation="h", marker_color=colors, showlegend=False,
            text=[f"{value:.3f}" if value > -10 else f"{value:.0f}" for value in frame["r2"].to_numpy()],
            textposition="outside", cliponaxis=False,
            hovertemplate="%{y}<br>" + setting + "<br>macro R2=%{text}<extra></extra>",
        ),
        row=1, col=column,
    )
    figure.update_xaxes(range=span, title_text="macro R2", row=1, col=column)
    figure.add_vline(x=0, line_color="#1F2937", line_width=1, row=1, col=column)
figure.update_yaxes(autorange="reversed")
figure.update_yaxes(showticklabels=False, row=1, col=2)
figure.update_yaxes(showticklabels=False, row=1, col=3)
figure.update_layout(
    title="The block that predicts is not the block that transfers",
    template="plotly_white", height=520, margin={"l": 200},
)
figure.show()

### Reading the three settings

**In-class, block ablation is almost uninformative.** Every block reaches macro
R2 0.988 or better on its own: the shape spectrum alone gets 0.9961, the physics
proxy alone 0.9881, and even v0 gets 0.9921 against the full v2 vector's 0.9998.
Predicting capacitance inside one family is easy enough that almost any faithful
description of the geometry suffices, so an ablation run only in this setting
would conclude, wrongly, that the blocks are interchangeable.

**Cross-class, the same blocks span three orders of magnitude**, from the
physical-metric block at 0.966 with ten labels down to the shape spectrum at
0.381. The ranking is not a rescaling of the in-class ranking.

**The physics proxy is the one block that predicts an unseen class with no
labels at all**, at **+0.661**, where every other block including the full vector
is negative. A two-dimensional boundary-element estimate is dimensionally
comparable across component classes in a way that shape descriptors are not - but
only once every family carries a correctly scaled ground plane, since the proxy
solves for charge at unit potential against that plane. On the earlier releases,
where CapN's plane was 500x too far away and TransmonCross had none, the same
block scored -6.06.

**The shape spectrum is the clearest reversal**, among the best in-class at
0.9961 and the worst cross-class at 0.381 with ten labels. Contour harmonics and
two-point correlations describe what a family's geometry *looks like*, and a comb
does not look like a cross.

The practical consequence: v2 should be published as blocks a user can select,
because the right subset depends on whether the task is prediction or transfer.

## 9. Does similarity mean anything across a class boundary?

Accuracy is not the only thing a foundation representation owes us. The routing
workflow needs cosine similarity to correlate with physical closeness even when
two designs come from different families. We sample cross-class pairs only, and
correlate their similarity against the gap in log mutual capacitance.

In [18]:
def similarity_table(matrix):
    reference = matrix.astype(np.float64)
    centre = reference.mean(axis=0)
    scale = reference.std(axis=0)
    keep = scale > 1e-8
    standardized = (reference[:, keep] - centre[keep]) / scale[keep]
    return standardized / np.maximum(np.linalg.norm(standardized, axis=1, keepdims=True), 1e-12)


log_mutual = data["log_mutual"].to_numpy()
names = sorted(CLASSES)
rows, samples = [], {}
for name, matrix in (("v0", v0), ("v2", v2)):
    unit = similarity_table(matrix)
    rng = np.random.default_rng(SEED)
    for index, first in enumerate(names):
        for second in names[index:]:
            left_pool = np.flatnonzero(components == first)
            right_pool = np.flatnonzero(components == second)
            left = rng.choice(left_pool, 20000)
            right = rng.choice(right_pool, 20000)
            keep = left != right
            left, right = left[keep], right[keep]
            similarity = np.sum(unit[left] * unit[right], axis=1)
            gap = np.abs(log_mutual[left] - log_mutual[right])
            label = f"within {first}" if first == second else f"{first} vs {second}"
            samples[(name, label)] = (similarity, gap)
            rows.append(
                {
                    "representation": name,
                    "pair": label,
                    "spearman": round(float(spearmanr(similarity, gap).statistic), 3),
                    "median cosine": round(float(np.median(similarity)), 3),
                }
            )
similarity_frame = pd.DataFrame(rows)
print("Spearman(cosine similarity, |difference in log mutual C|)")
print("negative means similar-looking designs really do behave similarly")
print()
similarity_frame.pivot(index="pair", columns="representation", values="spearman")

Spearman(cosine similarity, |difference in log mutual C|)
negative means similar-looking designs really do behave similarly



representation,v0,v2
pair,,
CapNInterdigitalTee vs GeneralizedCapNInterdigital,0.321,-0.490
CapNInterdigitalTee vs TransmonCross,-0.544,0.003
GeneralizedCapNInterdigital vs TransmonCross,0.309,0.224
within CapNInterdigitalTee,-0.278,-0.685
within GeneralizedCapNInterdigital,-0.509,-0.606
within TransmonCross,-0.199,-0.223


In [19]:
# %% hide input
order = [f"within {name}" for name in names] + [
    f"{names[i]} vs {names[j]}" for i in range(len(names)) for j in range(i + 1, len(names))
]
figure = go.Figure()
for name in ("v0", "v2"):
    frame = similarity_frame.query("representation == @name").set_index("pair").reindex(order)
    figure.add_trace(
        go.Bar(
            x=order,
            y=frame["spearman"].to_numpy(),
            name=name,
            marker_color=PALETTE[name],
            text=[f"{value:+.2f}" for value in frame["spearman"].to_numpy()],
            textposition="outside",
            hovertemplate="%{x}<br>" + name + "<br>Spearman=%{y:+.3f}<extra></extra>",
        )
    )
figure.add_hline(y=0, line_color="#1F2937", line_width=2)
figure.add_annotation(
    x=0.02, y=-0.78, xref="paper", yref="y", text="useful: similar looks, similar physics",
    showarrow=False, font={"color": "#2A9D8F"},
)
figure.add_annotation(
    x=0.02, y=0.42, xref="paper", yref="y", text="misleading: similar looks, different physics",
    showarrow=False, font={"color": "#D1495B"},
)
figure.update_layout(
    title="Similarity is trustworthy inside a class; across the qubit boundary it is not",
    yaxis={"title": "Spearman(cosine, |delta log C|)", "range": [-0.85, 0.5]},
    xaxis={"tickangle": -18},
    barmode="group",
    template="plotly_white",
    height=560,
)
figure.show()

## 10. What the three families actually look like

Every number above rests on the claim that one encoder reads three genuinely
different devices. It is worth seeing them. The panels below are drawn directly
from the GDS polygons of one representative design per family, coloured by the
role the encoder assigns each layer.

The `CapNInterdigitalTee` and `TransmonCross` files are the regenerated
port-complete exports; `GeneralizedCapNInterdigital` is the published sweep,
which already carried ordered ports. All three therefore expose the same
`2/0` and `3/0` marker convention, which is what lets terminal 0 and terminal 1
mean the same thing across families.

In [20]:
import shapely
from squadds.layouts.geometry_v2 import _role_geometry, _terminals, read_layer_geometry

ROLE_COLORS = {"conductor": "#00798C", "etch": "#E9C46A", "port": "#D1495B", "domain": "#C7CDD4"}


def build_gds_index():
    """Map design_id to a GDS source, preferring the port-complete regeneration.

    Published files are resolved lazily through the hub rather than assumed to be
    in a local snapshot, and any file that cannot be fetched is skipped instead of
    aborting the notebook.  The layouts repository advertises more
    GeneralizedCapNInterdigital artifacts than it actually stores.
    """
    index = {}
    published = pd.read_parquet(
        hf_hub_download("SQuADDS/SQuADDS_Layouts", "metadata/manifest.parquet", repo_type="dataset")
    )
    for row in published.itertuples():
        index[row.design_id] = ("published", row.gds_path)
    port_root = Path(PORT_COMPLETE_ROOT)
    manifest = port_root / "metadata/manifest.parquet"
    if manifest.is_file():
        for row in pd.read_parquet(manifest).itertuples():
            index[row.design_id] = ("port-complete", str(port_root / row.gds_path))
    return index


GDS_INDEX = build_gds_index()
_RESOLVED = {}


def resolve_gds(design_id):
    """Return (origin, path) for a design, or None when the file is unavailable."""
    if design_id in _RESOLVED:
        return _RESOLVED[design_id]
    entry = GDS_INDEX.get(design_id)
    result = None
    if entry is not None:
        origin, location = entry
        try:
            if origin == "port-complete":
                path = Path(location)
                result = (origin, path) if path.is_file() else None
            else:
                result = (origin, Path(hf_hub_download("SQuADDS/SQuADDS_Layouts", location, repo_type="dataset")))
        except Exception:  # noqa: BLE001 - advertised-but-absent artifacts are expected
            result = None
    _RESOLVED[design_id] = result
    return result


def polygon_traces(shape, name, color, *, opacity=0.75, show=True, paper="#FFFFFF"):
    traces, first = [], True
    for polygon in getattr(shape, "geoms", [shape]):
        if polygon.geom_type != "Polygon":
            continue
        x, y = polygon.exterior.xy
        traces.append(go.Scatter(
            x=list(x), y=list(y), mode="lines", fill="toself", fillcolor=color, opacity=opacity,
            line={"color": color, "width": 1.0}, name=name, legendgroup=name,
            showlegend=show and first, hoverinfo="skip",
        ))
        first = False
        for interior in polygon.interiors:
            hx, hy = interior.xy
            traces.append(go.Scatter(
                x=list(hx), y=list(hy), mode="lines", fill="toself", fillcolor=paper,
                line={"color": color, "width": 0.6}, legendgroup=name, showlegend=False, hoverinfo="skip",
            ))
    return traces


def role_traces(design_id, *, show_legend=False):
    resolved = resolve_gds(design_id)
    if resolved is None:
        return [], "unavailable", 0, 0
    origin, path = resolved
    grouped = _role_geometry(read_layer_geometry(path), None)
    traces = []
    for role in ("domain", "etch", "conductor", "port"):
        for _, shape in grouped[role]:
            traces.extend(polygon_traces(shape, role, ROLE_COLORS[role], show=show_legend))
    conductor = shapely.union_all([shape for _, shape in grouped["conductor"]])
    return traces, origin, len(_terminals(conductor, grouped["port"])), len(grouped["port"])


representatives = {}
for component in sorted(CLASSES):
    subset = data[data.component_name == component].reset_index(drop=True)
    order = [len(subset) // 2, *range(len(subset))]
    for position in order:
        candidate = subset.iloc[position]["design_id"]
        if resolve_gds(candidate) is not None:
            representatives[component] = candidate
            break

summary = []
for component, design_id in representatives.items():
    _, origin, terminals, ports = role_traces(design_id)
    summary.append({"component": component, "gds source": origin, "terminals": terminals, "port markers": ports})
pd.DataFrame(summary)

,component,gds source,terminals,port markers
0,CapNInterdigitalTee,port-complete,2,2
1,GeneralizedCapNInterdigital,published,2,2
2,TransmonCross,port-complete,2,2


In [21]:
# %% hide input
figure = make_subplots(rows=1, cols=3, subplot_titles=sorted(CLASSES), horizontal_spacing=0.06)
for column, component in enumerate(sorted(CLASSES), start=1):
    traces, origin, terminals, ports = role_traces(representatives[component], show_legend=column == 1)
    for trace in traces:
        figure.add_trace(trace, row=1, col=column)
    figure.update_xaxes(title_text="x (um)", row=1, col=column)
    figure.update_yaxes(scaleanchor=f"x{'' if column == 1 else column}", scaleratio=1, row=1, col=column)
figure.update_yaxes(title_text="y (um)", row=1, col=1)
figure.update_layout(
    title="One design from each family, coloured by the role universal-geometry-v2 assigns",
    template="plotly_white", height=470,
)
figure.show()

## 11. The closest and farthest shapes, by cosine similarity

Section 9 measured the similarity metric as a correlation. This section shows it
as geometry. For each of the six family pairings - three within a family and
three across - we find the pair of designs with the **highest** cosine similarity
and the pair with the **lowest**, and draw all four.

Use the slider to change pairing. The question to ask of each panel is whether
the pair the metric calls closest actually looks like it should behave alike, and
whether the pair it calls farthest really is unrelated.

In [22]:
reference = v2.astype(np.float64)
centre, scale = reference.mean(axis=0), reference.std(axis=0)
keep = scale > 1e-8
standardized = (reference[:, keep] - centre[keep]) / scale[keep]
UNIT = standardized / np.maximum(np.linalg.norm(standardized, axis=1, keepdims=True), 1e-12)

names = sorted(CLASSES)
rng = np.random.default_rng(SEED)
extremes, mean_cosine = {}, np.zeros((len(names), len(names)))
for i, first in enumerate(names):
    for j, second in enumerate(names):
        if j < i:
            continue
        left_pool = np.flatnonzero(components == first)
        right_pool = np.flatnonzero(components == second)
        left = rng.choice(left_pool, 40000)
        right = rng.choice(right_pool, 40000)
        valid = left != right
        left, right = left[valid], right[valid]
        similarity = np.sum(UNIT[left] * UNIT[right], axis=1)
        mean_cosine[i, j] = mean_cosine[j, i] = float(similarity.mean())
        # Walk inwards from each extreme until both endpoints can actually be
        # drawn.  Only a handful of files are resolved, not the whole catalogue.
        order = np.argsort(similarity)
        entry = {}
        for label, candidates in (("closest", order[::-1]), ("farthest", order)):
            for position in candidates[:40]:
                a, b = int(left[position]), int(right[position])
                if resolve_gds(data.iloc[a]["design_id"]) and resolve_gds(data.iloc[b]["design_id"]):
                    entry[label] = (a, b, float(similarity[position]))
                    break
            else:
                position = int(candidates[0])
                entry[label] = (int(left[position]), int(right[position]), float(similarity[position]))
        extremes[(first, second)] = entry

rows = []
for (first, second), entry in extremes.items():
    label = f"within {first}" if first == second else f"{first} vs {second}"
    rows.append({
        "pairing": label,
        "max cosine": round(entry["closest"][2], 4),
        "min cosine": round(entry["farthest"][2], 4),
        "mean cosine": round(mean_cosine[names.index(first), names.index(second)], 4),
    })
pd.DataFrame(rows)

,pairing,max cosine,min cosine,mean cosine
0,within CapNInterdigitalTee,0.9994,0.0222,0.7235
1,CapNInterdigitalTee vs GeneralizedCapNInterdig...,0.3439,-0.5597,-0.1956
2,CapNInterdigitalTee vs TransmonCross,0.3662,0.0117,0.2007
3,within GeneralizedCapNInterdigital,0.9924,-0.5831,0.2261
4,GeneralizedCapNInterdigital vs TransmonCross,0.0449,-0.7021,-0.4346
5,within TransmonCross,1.0000,0.6558,0.9075


In [23]:
# %% hide input
pairings = list(extremes)
figure = make_subplots(
    rows=1, cols=4,
    subplot_titles=["closest: design A", "closest: design B", "farthest: design A", "farthest: design B"],
    horizontal_spacing=0.035,
)
counts = []
for first, second in pairings:
    entry = extremes[(first, second)]
    before = len(figure.data)
    for column, (row_index, _kind) in enumerate(
        [(entry["closest"][0], "c"), (entry["closest"][1], "c"),
         (entry["farthest"][0], "f"), (entry["farthest"][1], "f")], start=1
    ):
        traces, _, _, _ = role_traces(data.iloc[row_index]["design_id"])
        for trace in traces:
            figure.add_trace(trace, row=1, col=column)
    counts.append(len(figure.data) - before)

start = 0
steps = []
for index, (first, second) in enumerate(pairings):
    visible = [False] * len(figure.data)
    for offset in range(counts[index]):
        visible[start + offset] = True
    start += counts[index]
    entry = extremes[(first, second)]
    label = f"within {first[:14]}" if first == second else f"{first[:12]} vs {second[:12]}"
    steps.append({
        "label": label, "method": "update",
        "args": [{"visible": visible},
                 {"title": f"{label}   closest cosine {entry['closest'][2]:.4f}   "
                           f"farthest cosine {entry['farthest'][2]:.4f}"}],
    })
for index in range(len(figure.data)):
    figure.data[index].visible = index < counts[0]
first, second = pairings[0]
entry = extremes[(first, second)]
for column in range(1, 5):
    figure.update_yaxes(scaleanchor=f"x{'' if column == 1 else column}", scaleratio=1, row=1, col=column)
figure.update_layout(
    title=f"within {first[:14]}   closest cosine {entry['closest'][2]:.4f}   "
          f"farthest cosine {entry['farthest'][2]:.4f}",
    template="plotly_white", height=430, showlegend=False,
    sliders=[{"active": 0, "currentvalue": {"prefix": "pairing: "}, "pad": {"t": 68}, "steps": steps}],
)
figure.show()

In [24]:
# %% hide input
figure = go.Figure(go.Heatmap(
    z=mean_cosine, x=names, y=names, colorscale="Viridis", zmid=0,
    text=[[f"{value:.3f}" for value in row] for row in mean_cosine],
    texttemplate="%{text}", colorbar={"title": "mean cosine"},
    hovertemplate="%{y}<br>%{x}<br>mean cosine=%{z:.4f}<extra></extra>",
))
figure.update_layout(
    title="Mean standardized cosine similarity between and within families",
    template="plotly_white", height=470, margin={"l": 210, "b": 150},
)
figure.show()

### Reading the shapes

The mean-cosine matrix is now strongly diagonal: 0.72 within `CapNInterdigitalTee`,
0.91 within `TransmonCross`, and 0.23 within `GeneralizedCapNInterdigital`,
against -0.20 and -0.44 for the two cross-family pairs involving the generalized
coupler. Designs resemble their own family far more than another.

The slider shows what that means geometrically. Within a family the closest pair
reaches cosine 0.999 and is typically two designs differing in one swept
dimension, while the farthest pair sits at opposite ends of the sweep.

Across families the maxima are much lower: 0.343 between the two coupler
families and only **0.044** between the generalized coupler and the transmon. A
transmon cross and an interdigital comb share almost no shape vocabulary, so what
the metric ranks across that boundary is overall scale and metal fraction rather
than anything that sets the capacitance. That is the same weakness the held-out
`TransmonCross` rotation reports as a number, seen directly.

## 12. What this experiment establishes

This run uses the **unified** layout release. All three families now share one
convention: a ground plane at a fixed 169 um per-side margin and centred on it,
the etch expressed as a single hole in that plane rather than as its own layer,
and two ordered ports that bridge the moat from each terminal to ground. Only the
`GeneralizedCapNInterdigital` rows come from the published sweep; the other two
were regenerated. Joining them is legitimate only because
`universal-geometry-v2` consults no catalogue statistics.

**Established**

- Three SQuADDS families share a mutual-capacitance target, and across them the
  design-option vocabularies intersect in exactly one name, `orientation`, a
  placement angle. No parameter-schema baseline exists for a three-class model.
- Cross-family transfer into `CapNInterdigitalTee` is close to saturated: 13
  labeled designs reach macro R2 0.994, against 0.769 for v0 and 0.625 for the
  aligned parameter baseline.
- Held-out `CapNInterdigitalTee` reaches **+0.824 with zero labels**.
- On a class-balanced cohort **two of three rotations are positive**, at +0.596
  and +0.544.
- A brand-new component family needs roughly ten labels, reaching 0.945.
- The boundary-element physics proxy is the only block that predicts an unseen
  class with no labels at all, at **+0.661**, and it only does so once every
  family carries a correctly scaled ground plane.
- A source prior helps only when the representation aligns the classes: at five
  labels it lifts v2 from 0.629 to 0.894 and pushes v0 from 0.261 to -0.502.
- Unifying the ground and port convention was worth a large repair. Held-out
  `GeneralizedCapNInterdigital` moved from -3.998 on the mismatched release to
  -0.001, and averaged zero-shot from -2.58 to -0.627.

**Not established, and worth stating plainly**

- **`TransmonCross` is still not predictable from the other two**, at -5.385 on
  the full catalogue and -2.961 balanced. It is negative in both cohorts, so this
  is not a class-size artefact. It is also the family whose two terminals differ
  most in scale, a large cross against a small claw, and the family with the
  lowest cross-family cosine, 0.044 against the generalized coupler.
- Zero-shot cross-class prediction is close to break-even, not reliable. One of
  three rotations on the full catalogue is clearly positive.
- Ordered ports alone changed nothing measurable on `TransmonCross`. Its
  conductor geometry was byte-identical to the published release and its port
  ordering never disagreed with the previous area-based fallback in any of 1,934
  files. The gain here came from adding a ground plane and a moat, not from the
  markers.
- The physics-proxy residual target remains worse than predicting capacitance
  directly in every rotation.
- Block ablation run only in-class is misleading. Every v2 block predicts
  capacitance inside a single family at macro R2 0.988 or better, so that setting
  cannot distinguish them; cross-class the same blocks span 0.966 down to 0.381.
- Cross-class similarity is still not trustworthy: the largest cross-family
  cosine is 0.343 for the two coupler families and 0.044 between the generalized
  coupler and the transmon.
- All three classes remain electrostatic; nothing here shows transfer to the
  eigenmode quantities `CavityClawRouteMeander` reports.

**What we would do next**

Regenerate `GeneralizedCapNInterdigital` with the same unified exporter so all
three families come from one pipeline rather than two. Then attack
`TransmonCross` directly: it is the only rotation still failing, and the
terminal-scale asymmetry hypothesis is now specific enough to test by
conditioning the coupling spectrum on per-terminal scale.